# Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


from sklearn.linear_model import LogisticRegression

I (Keith) use sklearn: 1.4.2 imblearn: 0.12.3

In [27]:
import sklearn, imblearn
print("sklearn:", sklearn.__version__, "imblearn:", imblearn.__version__)

sklearn: 1.2.2 imblearn: 0.11.0


# Keep this random state

In [2]:
RANDOM_STATE=42

# Splitting functions and custom IQRClipper

In [3]:
# Convert one-hot cohort flags into a single label
def label_subcohort(df):
    df = df.copy()
    df["cohort"] = np.select(
        [
            df["is_mi"] == 1,
            df["is_pcr"] == 1,
            df["is_cbs"] == 1
        ],
        ["MI", "PCI", "CABG"],
        default="Other"
    )
    return df

# 1) collapse race to a few stable bins
def collapse_race(r):
    if pd.isna(r): return "UNKNOWN/OTHER"
    r = str(r).upper()
    if r.startswith("WHITE"): return "WHITE"
    if r.startswith("BLACK"): return "BLACK"
    if r.startswith("ASIAN"): return "ASIAN"
    if "HISPANIC" in r:        return "HISPANIC"
    if "PORTUGUESE" in r:      return "PORTUGUESE"
    if "SOUTH AMERICAN" in r:  return "HISPANIC"  # sensible merge
    if "PACIFIC ISLANDER" in r:return "PACIFIC"
    if r in {"OTHER","UNKNOWN","UNABLE TO OBTAIN","PATIENT DECLINED TO ANSWER",
             "MULTIPLE RACE/ETHNICITY","AMERICAN INDIAN/ALASKA NATIVE",
             "WHITE - OTHER EUROPEAN","WHITE - EASTERN EUROPEAN","WHITE - RUSSIAN",
             "WHITE - BRAZILIAN","BLACK/CAPE VERDEAN","BLACK/CARIBBEAN ISLAND"}:
        return "UNKNOWN/OTHER"
    return "UNKNOWN/OTHER"

def build_subject_strata(df: pd.DataFrame,
                         min_bin_size: int = 5,
                         outcome_col: str = "y",
                         cohort_col: str = "cohort",
                         race_col: str = "race"):
    
    g = df.copy()
    
    g["race_collapsed"] = g[race_col].map(collapse_race)

    # Make a *stable* per-subject cohort label.
    # Option A: majority cohort across their stays
    subj_mode_cohort = (
        g.groupby(["subject_id", cohort_col]).size()
         .reset_index(name="n")
         .sort_values(["subject_id","n"], ascending=[True, False])
         .drop_duplicates("subject_id")[["subject_id", cohort_col]]
         .rename(columns={cohort_col: "cohort_major"})
    )

    # Outcome as a subject-level flag (any positive stay)
    subj_y = (
        g.groupby("subject_id")[outcome_col]
         .max()
         .rename("y_any")
         .reset_index()
    )

    # Subject-level collapsed race: mode
    subj_race = (
        g.groupby(["subject_id","race_collapsed"]).size()
         .reset_index(name="n")
         .sort_values(["subject_id","n"], ascending=[True, False])
         .drop_duplicates("subject_id")[["subject_id","race_collapsed"]]
    )

    subj = (
        subj_mode_cohort.merge(subj_race, on="subject_id")
                        .merge(subj_y, on="subject_id")
    )

    # Compose strata label; you can add age bands, etc., if you want more control
    subj["strata"] = (
        subj["cohort_major"].astype(str) + "|" +
        subj["race_collapsed"].astype(str) + "|" +
        subj["y_any"].astype(str)
    )

    # Merge rare strata into a single bin to avoid stratify errors
    counts = subj["strata"].value_counts()
    rare = counts[counts < min_bin_size].index
    subj.loc[subj["strata"].isin(rare), "strata"] = "OTHER_BIN"

    return subj[["subject_id","strata"]]

def remove_outlier_drop_gapdays_and_split(df_all: pd.DataFrame,
                                          val_and_test_size: float = 0.30,
                                          random_state: int = 42,
                                          outcome_col: str = "y",
                                          cohort_col: str = "cohort",
                                          race_col: str = "race"):
    """
    - Hard-remove obvious data-entry outliers.
    - Drop prev_gap_days (non-imputable by design).
    - Stratified split by *subject_id* using a patient-level strata label
      (cohort_major × collapsed race × y_any), with rare bins merged.
    - Returns row-level train/val/test DataFrames.
    """
    
    
    df = df_all.copy()
    if "cohort" not in df.columns:
        df = label_subcohort(df)

    # 0) hard outliers + drop prev_gap_days
    df = df[(df["glucose_mean"] < 999999) & (df["wbc_last_24h"] < 400)]
    if "prev_gap_days" in df.columns:
        df = df.drop(columns=["prev_gap_days"])

    # 1) build subject-level strata
    subj = build_subject_strata(df, min_bin_size=5,
                                outcome_col=outcome_col,
                                cohort_col=cohort_col,
                                race_col=race_col)

    # 2) split BY subject IDs into train vs temp (val+test) with stratify
    train_ids, temp_ids = train_test_split(
        subj["subject_id"].values,
        test_size=val_and_test_size,
        random_state=random_state,
        stratify=subj["strata"].values
    )

    # 3) from the temp pool, split val vs test (50/50) with stratify again
    temp = subj[subj["subject_id"].isin(temp_ids)]
    val_ids, test_ids = train_test_split(
        temp["subject_id"].values,
        test_size=0.5,
        random_state=random_state,
        stratify=temp["strata"].values
    )

    # 4) map IDs back to rows
    train_df = df[df["subject_id"].isin(train_ids)].copy()
    val_df   = df[df["subject_id"].isin(val_ids)].copy()
    test_df  = df[df["subject_id"].isin(test_ids)].copy()

    # Optional sanity checks
    for s in ["train","val","test"]:
        print(s, df[df.subject_id.isin(eval(f"{s}_ids"))][outcome_col].mean())

    return train_df, val_df, test_df


def examine_splits(df, df_type='train'):
    print('-'*25, f'EXAMINING {df_type} data', '-'*25)
    print(f'{df_type} data has {df.shape[0]} ICU stays across {df.subject_id.nunique()} patients.')
    print('-'*25, 'EXAMINE FIRST FEW INSTANCES', '-'*25)
    display(df.head())
    
    print('-'*25, 'CLASS DISTRIBUTION (%)', '-'*25)
    display(df.y.value_counts(normalize=True)*100)
    
    print('-'*25, f'RACE DISTRIBUTION (%) ({df.race.nunique()} races)', '-'*25)
    display(df.race.value_counts(normalize=True)*100)
    
    print('-'*25, 'COHORT DISTRIBUTION (%)', '-'*25)
    display(df.cohort.value_counts(normalize=True)*100)
    
    print('-'*25, 'AGE DISTRIBUTION', '-'*25)
    df.hist('age')


class IQRClipper(BaseEstimator, TransformerMixin):
    """
    Clips feature values based on the Interquartile Range (IQR).
    Values below Q1 - factor*IQR are set to that lower bound,
    and values above Q3 + factor*IQR are set to that upper bound.

    Parameters
    ----------
    factor : float, default=1.5
        Multiplier for the IQR range.
    """

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        Q1 = X_df.quantile(0.25)
        Q3 = X_df.quantile(0.75)
        IQR = Q3 - Q1
        self.lower_bounds_ = Q1 - self.factor * IQR
        self.upper_bounds_ = Q3 + self.factor * IQR
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        X_clipped = X_df.clip(lower=self.lower_bounds_, upper=self.upper_bounds_, axis=1)
        # Return as same type as input
        return X_clipped if isinstance(X, pd.DataFrame) else X_clipped.to_numpy()

# REMOVE OBVIOUS OUTLIERS, DROP COLUMNS, THEN SPLIT

In [4]:
# REMOVE OBVIOUS OUTLIERS, DROP COLUMNS, THEN SPLIT

df = pd.read_csv("merged_chd_model_v2.csv")

train_df, val_df, test_df = remove_outlier_drop_gapdays_and_split(
    df,
    val_and_test_size = 0.3,
    random_state=RANDOM_STATE,
    outcome_col='y',
    cohort_col='cohort',
    race_col='race'
)

train 0.058043273753527753
val 0.06519823788546256
test 0.05415002219263205


Run this to see the distribution of train, val, and test splits

In [12]:
# examine_splits(train_df, df_type='train')
# examine_splits(val_df, df_type='val')
# examine_splits(test_df, df_type='test')

# 0. DROP IDENTIFIERS AND PREP THE DATA FOR HYPERPARAMETER TUNING

In [6]:
# 0. DROP IDENTIFIERS AND PREP THE DATA FOR HYPERPARAMETER TUNING

# Columns to drop from X
ID_COLS = ["subject_id", "hadm_id", "stay_id", "is_mi", "is_pcr", "is_cbs"]
DROP_COLS = [c for c in (ID_COLS + ["y"]) if c in train_df.columns]

# Build X_all / y_all and derive feature lists FROM X_all
X_all = pd.concat([train_df, val_df], axis=0).reset_index(drop=True).drop(columns=DROP_COLS)
y_all = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)["y"]


# define numeric / categorical feature sets
numeric_cols = X_all.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_all.columns if c not in numeric_cols]

# 1. DEFINE THE PIPELINE (PREPROCESSING & OVERSAMPLING & MODELLING)

To swap models, change the final pipeline step from LogisticRegression to your own estimator (e.g. MLP via scikeras). Keep preprocessing identical.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, average_precision_score,
    roc_auc_score, classification_report
)


num_pipe = Pipeline([
    ("iqr_clip", IQRClipper(factor=1.5)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ],
    remainder="drop"
)


X_train = train_df.drop(columns=ID_COLS + ["y"])
y_train = train_df["y"]

X_val = val_df.drop(columns=ID_COLS + ["y"])
y_val = val_df["y"]



want_oversample = True  # True=with resampling; False=without resampling

steps = [("pre", preprocessor)]
if want_oversample:
    steps.append(("ros", RandomOverSampler(random_state=RANDOM_STATE)))

# neg = (y_train == 0).sum()
# pos = (y_train == 1).sum()
# spw = neg / max(pos, 1)


X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)


model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.9,
    min_child_weight=1.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    scale_pos_weight=None,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_val_processed, y_val)],
    verbose=True
)


from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


pipe = ImbPipeline(steps=[
    ("pre", preprocessor),
    ("ros", RandomOverSampler(random_state=RANDOM_STATE)),
    ("model", model)
])


pipe.fit(X_train, y_train)



eval_result = model.evals_result()


for i in range(1, model.get_params()["n_estimators"] + 1):
    y_pred_proba = model.predict_proba(X_train_processed, iteration_range=(0, i))[:, 1]
    y_pred_label = (y_pred_proba >= 0.5).astype(int)

    acc = accuracy_score(y_train, y_pred_label)
    auc = eval_result["validation_0"]["auc"][i - 1]
    aucpr = eval_result["validation_0"]["aucpr"][i - 1]
    tn, fp, fn, tp = confusion_matrix(y_train, y_pred_label).ravel()

    print(f"Round {i:03d}: "
          f"Acc = {acc:.4f} | "
          f"AUC = {auc:.4f} | "
          f"AUC-PR = {aucpr:.4f} | "
          f"TP = {tp}, FP = {fp}, TN = {tn}, FN = {fn}")




[0]	validation_0-auc:0.58669	validation_0-aucpr:0.10247
[1]	validation_0-auc:0.60503	validation_0-aucpr:0.11070
[2]	validation_0-auc:0.64966	validation_0-aucpr:0.13780
[3]	validation_0-auc:0.65472	validation_0-aucpr:0.13471
[4]	validation_0-auc:0.67168	validation_0-aucpr:0.15312
[5]	validation_0-auc:0.68245	validation_0-aucpr:0.16977
[6]	validation_0-auc:0.69156	validation_0-aucpr:0.18332
[7]	validation_0-auc:0.70819	validation_0-aucpr:0.17415
[8]	validation_0-auc:0.71911	validation_0-aucpr:0.17522
[9]	validation_0-auc:0.71648	validation_0-aucpr:0.16681
[10]	validation_0-auc:0.71433	validation_0-aucpr:0.17291
[11]	validation_0-auc:0.71834	validation_0-aucpr:0.17219
[12]	validation_0-auc:0.72077	validation_0-aucpr:0.17305
[13]	validation_0-auc:0.71935	validation_0-aucpr:0.17905
[14]	validation_0-auc:0.72367	validation_0-aucpr:0.18527
[15]	validation_0-auc:0.72297	validation_0-aucpr:0.18039
[16]	validation_0-auc:0.72728	validation_0-aucpr:0.18340
[17]	validation_0-auc:0.72517	validation_

# 2. HYPERPARAMETER TUNING
 ALTHOUGH WE DO NOT USE CV, GridSearchCV IS USED FOR EASIER CODE MANAGEMENT AND MODULARITY

# 3. EVALUATE GENERALISATION PERFORMANCE (feel free to change this)

In [ ]:
# =========================
# 3) EVALUATE (on held-out test)
# =========================
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
    accuracy_score
)

# ==== 9. Test Set Evaluation ====
DROP_COLS_TEST = [c for c in (ID_COLS + ["y"]) if c in test_df.columns]
Xte = test_df.drop(columns=DROP_COLS_TEST)
yte = test_df["y"].astype(int)

proba_te = pipe.predict_proba(Xte)[:, 1]
pred_te  = (proba_te >= 0.2).astype(int)

acc     = accuracy_score(yte, pred_te)
auc     = roc_auc_score(yte, proba_te)
aucpr   = average_precision_score(yte, proba_te)
clf_rep = classification_report(yte, pred_te, digits=3)

print("======  Test Set Evaluation ======")
print(f"[TEST] Accuracy  : {acc:.4f}")
print(f"[TEST] ROC-AUC   : {auc:.4f}")
print(f"[TEST] PR-AUC    : {aucpr:.4f}")
print("\n[TEST] Classification Report:")
print(clf_rep)


====== 📊 Test Set Evaluation ======
[TEST] Accuracy  : 0.7133
[TEST] ROC-AUC   : 0.7232
[TEST] PR-AUC    : 0.1781

[TEST] Classification Report:
              precision    recall  f1-score   support

           0      0.970     0.719     0.826      2131
           1      0.110     0.607     0.186       122

    accuracy                          0.713      2253
   macro avg      0.540     0.663     0.506      2253
weighted avg      0.923     0.713     0.791      2253

